# Step 1 · Access checks and imported Xenium previews

This is a clean retry copy of your attached Step1 notebook. It uses your existing
`workflow_config.json`, manifest, and imported SpatialData stores, including
`xenium_vanilla_cell`. The original preview code and channel/ROI settings are retained.

The reported failure was an operating-system denial while Python read
`datashader/resampling.py`, **before a sample store was opened**. The earlier
`functools.partial` warnings are separate. There is no reason from this traceback
to repeat Step0B.

## Before running

1. Preserve any unsaved edits/error text from the old notebook before closing it.
2. Put this notebook and `step1_runtime_check.py` beside your existing
   `xenium_workflow/` folder and `workflow_config.json`.
3. Use a fresh kernel from the same existing uv environment. Restart only the
   Step1 kernel; a restart cannot itself fix a persistent filesystem restriction.
4. Run the access checks, then the dependency-import check, before the previews.

The helper does not change permissions or install packages. Its optional write
probes create and remove two tiny, uniquely named temporary files in each tested
folder. They do not write to a Xenium store or to an existing notebook.

A kernel-side probe does **not** test Jupyter's server-side Save operation. If
saving still fails, also try saving a new small text file through JupyterLab in
the same folder and retain the exact Save error.


In [1]:
from pathlib import Path
import sys
import json

# Reduce traceback source inspection while diagnosing inaccessible package files.
# This changes error presentation only; it does not repair filesystem access.
try:
    _shell = get_ipython()
except NameError:
    _shell = None
if _shell is not None:
    _shell.run_line_magic("xmode", "Minimal")

# If the notebook is launched from another directory, set this explicitly.
PROJECT_ROOT_OVERRIDE = None  # Example: Path("/path/to/xenium_initial_processing")
if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().absolute()
else:
    PROJECT_ROOT = None
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        try:
            if (_candidate / "xenium_workflow").is_dir():
                PROJECT_ROOT = _candidate
                break
        except OSError as exc:
            print("Cannot inspect candidate folder:", _candidate, type(exc).__name__, str(exc))
if PROJECT_ROOT is None:
    raise RuntimeError("Set PROJECT_ROOT_OVERRIDE to the existing xenium_initial_processing folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from step1_runtime_check import (
    collect_access_report, print_access_report, save_access_report,
    require_access, import_preview_dependencies, error_details,
)
print("Current kernel:", sys.executable)
print("Project folder:", PROJECT_ROOT)


Exception reporting mode: Minimal
Current kernel: /home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python
Project folder: /home/domino/reny28/Projects/Xenium_projects/TBIO-8136/code/xenium_initial_processing


In [2]:
RUN_TEMPORARY_WRITE_PROBES = True
access_report = collect_access_report(PROJECT_ROOT, check_writes=RUN_TEMPORARY_WRITE_PROBES)
print_access_report(access_report)

# A separate small JSON can survive even if saving the notebook later fails.
# If this folder is also unwritable, the report remains visible above.
try:
    diagnostic_path = save_access_report(access_report, PROJECT_ROOT)
    print("Diagnostic JSON:", diagnostic_path)
except OSError as exc:
    print("Could not save diagnostic JSON:", json.dumps(error_details(exc)))

require_access(access_report)
print("Access checks passed. Next, test imports in this same kernel.")


{
  "created_utc": "2026-11-06T18:56:27.942586+00:00",
  "python_executable": "/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python",
  "python_version": "3.13.7",
  "sys_prefix": "/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv",
  "project_root": "/home/domino/reny28/Projects/Xenium_projects/TBIO-8136/code/xenium_initial_processing",
  "effective_uid": 12574,
  "effective_gid": 12574,
  "versions": {
    "datashader": {
      "installed": "0.19.0",
      "expected": "0.19.0"
    },
    "spatialdata": {
      "installed": "0.7.2",
      "expected": "0.7.2"
    },
    "spatialdata-plot": {
      "installed": "0.3.3",
      "expected": "0.3.3"
    }
  },
  "source_checks": [
    {
      "path": "/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/datashader/__init__.py",
      "stat": {
        "mode": "-rw-r--r--",
        "uid": 12574,
        "gid": 12574,
        "bytes": 2332
 

## Read the access report before continuing

- **`ordinary_read` or `code_read` failed:** Python still cannot read the package
  source in the required way. Restore access to the exact path shown in the
  report. If ordinary reading works but code reading fails, retain that
  distinction for your platform/environment administrator.
- **A write probe failed:** inspect its directory and error. This may explain
  missing figure/log outputs. A failure beside the notebook is also useful
  evidence for the separate Save problem.
- **Checks pass:** continue with the actual imports below. These small access
  probes do not validate every dependency or the imported dataset.

See `README_Step1_Recovery.md` for recovery steps. The notebook deliberately
stops at a failed check; it cannot override operating-system access controls.


In [3]:
# This is the real import in the active kernel, before opening a sample.
dependency_imports = import_preview_dependencies()
print(json.dumps(dependency_imports, indent=2))


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:531: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left_exclusive = partial(_left_exclusive_join_spatialelement_table)
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:533: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you

RuntimeError: Import failed before opening a Xenium store (spatialdata): {"type": "PermissionError", "errno": 1, "filename": "/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/datashader/resampling.py", "message": "[Errno 1] Operation not permitted: '/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/datashader/resampling.py'"}. Restore package/runtime access; use a fresh kernel before retrying.

## Resume the existing Step1 preview workflow

The cells below are retained from your uploaded notebook. The same manifest,
channels, coordinate handling, overview generation, and optional ROI are used.
Figures and logs are written to your configured output folders; existing preview
PNGs with the same names can be replaced when you rerun their generation.

Compact error display stays enabled for this diagnostic run. After access is
restored, `%xmode Context` in a separate cell restores ordinary contextual
tracebacks if desired.


In [4]:
from pathlib import Path
import sys
import gc
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# PROJECT_ROOT was selected in the access-check cell above.
from xenium_workflow.config import WorkflowConfig, environment_report, atomic_json
from xenium_workflow.manifest import selected_samples, atomic_csv
from xenium_workflow.io import TABLE_KEY, read_sample
from xenium_workflow.reporting import log_event, log_exception, per_region_summary, merge_summary

PermissionError: [Errno 1] Operation not permitted: '/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/matplotlib/__init__.py'

In [5]:
config = WorkflowConfig.load(PROJECT_ROOT / "workflow_config.json")
samples = selected_samples(config)
display(samples[["sample_id", "sample_name", "run_id", "slide_id", "region_name"]])
print(f"Selected {len(samples)} technical regions; output: {config.root}")

NameError: name 'WorkflowConfig' is not defined

In [ ]:
from xenium_workflow.plotting import preview_spatialdata, centroid_map
from xenium_workflow.reporting import table_inventory
PREVIEW_CHANNEL = 0  # Usually DAPI; inspect channel coordinates below to confirm.

## 1. Inspect element names and morphology channels

In [ ]:
sdata = read_sample(samples.iloc[0]["zarr_path"])
display(table_inventory(sdata))
for key, image in sdata.images.items():
    array = next(iter(image["scale0"].data_vars.values())) if hasattr(image, "children") else image
    print(key, "channels:", array.coords["c"].values.tolist() if "c" in array.coords else "no channel axis")
del sdata
gc.collect()

## 2. Save morphology / cell-label overviews and centroid maps

Pyramid selection is automatic. Do not set `scale='full'` for a large whole-region image. The centroid map uses
micrometres; the SpatialData image/label overview displays its `global` pixel coordinates. The original metadata
may contain several stain channels; change `PREVIEW_CHANNEL` if channel 0 is not the desired one.

In [ ]:
for row in samples.to_dict("records"):
    sdata = None
    try:
        sdata = read_sample(row["zarr_path"])
        folder = config.root / "figures" / row["sample_id"]
        fig = preview_spatialdata(sdata, folder / "step1_morphology_and_labels.png", channel=PREVIEW_CHANNEL)
        plt.close(fig)
        fig = centroid_map(sdata.tables[TABLE_KEY], folder / "step1_centroids_um.png",
                           title=row["sample_name"] + " · native centroids")
        plt.close(fig)
        log_event(config.root, "step1_preview", row["sample_id"], "complete")
        print("Saved previews:", row["sample_id"], flush=True)
    except Exception as exc:
        log_exception(config.root, "step1_preview", row["sample_id"], exc)
        raise
    finally:
        del sdata
        gc.collect()

## 3. Review one saved overview

In [ ]:
from IPython.display import Image
display(Image(filename=str(config.root / "figures" / samples.iloc[0]["sample_id"] / "step1_morphology_and_labels.png")))

## 4. Optional ROI preview

Specify `(xmin, ymin, xmax, ymax)` in **native Xenium micrometres**, as in `obsm['spatial_um']`.
The helper converts the bounds to the reader's global pixels. `ROI_UM=None` skips this optional block.

In [ ]:
ROI_SAMPLE_ID = samples.iloc[0]["sample_id"]
ROI_UM = None  # Example only: (1000.0, 1000.0, 1500.0, 1500.0); choose a real tissue location.
if ROI_UM is not None:
    match = samples.loc[samples["sample_id"].eq(ROI_SAMPLE_ID)]
    if len(match) != 1:
        raise ValueError("ROI_SAMPLE_ID must identify one selected manifest row.")
    row = match.iloc[0].to_dict()
    sdata = read_sample(row["zarr_path"])
    try:
        fig = preview_spatialdata(sdata, config.root / "figures" / ROI_SAMPLE_ID / "step1_roi.png",
                                 roi_um=ROI_UM, channel=PREVIEW_CHANNEL, dpi=200)
        plt.show()
        plt.close(fig)
    finally:
        del sdata
        gc.collect()

Continue with Step2. H&E is not plotted because no registration is available. Sharing the source H&E image between
two regions does not provide the missing transforms.